In [1]:
import heapq

graph = {
    'S': [('A', 3), ('C', 2), ('D', 2)],
    'A': [],
    'B': [('E', 2)],
    'C': [('F', 1)],
    'D': [('B', 3), ('G', 8)],
    'E': [('G', 2)],
    'F': [('E', 0), ('G', 4)],
    'G': [],
}

def ucs(graph, start, goal):
   
    frontier = [(0, start, [start])]
    explored = []                      

    while frontier:
        cost, node, path = heapq.heappop(frontier)
        if node in explored:           
            continue
        explored.append(node)

        if node == goal:
            return explored, path, cost

        for neighbour, step_cost in graph[node]:
            if neighbour not in explored:
                heapq.heappush(frontier, (cost + step_cost, neighbour, path + [neighbour]))
    return explored, None, float('inf')

explored, path, cost = ucs(graph, 'S', 'G')
print("Explored set (in order):", explored)
print("Path found             :", " -> ".join(path))
print("Total path cost        :", cost)

Explored set (in order): ['S', 'C', 'D', 'A', 'F', 'E', 'B', 'G']
Path found             : S -> C -> F -> E -> G
Total path cost        : 5


In [2]:
maze_edges = [
    ('A', 'D'), ('D', 'B'), ('B', 'C'), ('B', 'F'), ('B', 'E'),
    ('D', 'G'), ('G', 'H'), ('H', 'I'), ('I', 'J'), ('I', 'K'),
    ('K', 'L'), ('K', 'M'), ('M', 'N'),
]

maze = {n: [] for n in 'ABCDEFGHIJKLMN'}
for a, b in maze_edges:
    maze[a].append((b, 1))
    maze[b].append((a, 1))

explored, path, cost = ucs(maze, 'A', 'M')
print("Explored set (in order):", explored)
print("Shortest path          :", " -> ".join(path))
print("Total cost (steps)     :", cost)

Explored set (in order): ['A', 'D', 'B', 'G', 'C', 'E', 'F', 'H', 'I', 'J', 'K', 'L', 'M']
Shortest path          : A -> D -> G -> H -> I -> K -> M
Total cost (steps)     : 6


In [3]:
import heapq

GOAL = (1, 2, 3,
        4, 5, 6,
        7, 8, 0)

START = (2, 1, 3,
         4, 7, 6,
         5, 8, 0)

def goal_test(state):
    return state == GOAL

def operators(state):
    """Return all states reachable by sliding one tile into the blank."""
    result = []
    b = state.index(0)
    row, col = divmod(b, 3)
    for dr, dc, name in [(-1, 0, 'Up'), (1, 0, 'Down'), (0, -1, 'Left'), (0, 1, 'Right')]:
        r, c = row + dr, col + dc
        if 0 <= r < 3 and 0 <= c < 3:
            s = list(state)
            n = r * 3 + c
            s[b], s[n] = s[n], s[b]
            result.append((tuple(s), name))
    return result

def h1(state):
    """Misplaced tiles."""
    return sum(1 for i, t in enumerate(state) if t != 0 and t != GOAL[i])

def h2(state):
    """Sum of Manhattan distances."""
    total = 0
    for i, t in enumerate(state):
        if t == 0:
            continue
        g = GOAL.index(t)
        total += abs(i // 3 - g // 3) + abs(i % 3 - g % 3)
    return total

def show(state):
    for i in range(0, 9, 3):
        print(' '.join(str(x) if x else '_' for x in state[i:i+3]))
    print()

In [4]:
def GreedySearch(state, heuristic=h2):
    """Greedy best-first search: always expands the state with the smallest h(n).
    Returns (moves, states, nodes_expanded), or None if no solution is found."""
    counter = 0                                   
    frontier = [(heuristic(state), counter, state, [], [state])]
    visited = set()
    expanded = 0

    while frontier:
        h, _, current, moves, states = heapq.heappop(frontier)
        if current in visited:
            continue
        visited.add(current)
        expanded += 1

        if goal_test(current):
            return moves, states, expanded

        for nxt, move in operators(current):
            if nxt not in visited:
                counter += 1
                heapq.heappush(frontier, (heuristic(nxt), counter, nxt, moves + [move], states + [nxt]))
    return None

In [5]:
print("Initial state:"); show(START)
print("Goal state:");    show(GOAL)
print("h1(start) =", h1(START), "| h2(start) =", h2(START))

Initial state:
2 1 3
4 7 6
5 8 _

Goal state:
1 2 3
4 5 6
7 8 _

h1(start) = 4 | h2(start) = 6


In [6]:
for name, h in [("h1 (misplaced tiles)", h1), ("h2 (Manhattan distance)", h2)]:
    moves, states, expanded = GreedySearch(START, h)
    print(f"=== Greedy Best-First Search with {name} ===")
    print("Solution length :", len(moves), "moves")
    print("Nodes expanded  :", expanded)
    print("Moves           :", moves)
    print()

=== Greedy Best-First Search with h1 (misplaced tiles) ===
Solution length : 30 moves
Nodes expanded  : 458
Moves           : ['Up', 'Left', 'Left', 'Up', 'Right', 'Down', 'Right', 'Down', 'Left', 'Up', 'Left', 'Down', 'Right', 'Right', 'Up', 'Up', 'Left', 'Down', 'Left', 'Up', 'Right', 'Right', 'Down', 'Left', 'Up', 'Left', 'Down', 'Right', 'Right', 'Down']

=== Greedy Best-First Search with h2 (Manhattan distance) ===
Solution length : 18 moves
Nodes expanded  : 284
Moves           : ['Left', 'Up', 'Up', 'Left', 'Down', 'Down', 'Right', 'Right', 'Up', 'Up', 'Left', 'Down', 'Left', 'Up', 'Right', 'Right', 'Down', 'Down']



In [7]:
# Step-by-step solution using the better heuristic (h2)
moves, states, expanded = GreedySearch(START, h2)
for i, s in enumerate(states):
    print("Start" if i == 0 else f"Step {i}: {moves[i-1]}")
    show(s)

Start
2 1 3
4 7 6
5 8 _

Step 1: Left
2 1 3
4 7 6
5 _ 8

Step 2: Up
2 1 3
4 _ 6
5 7 8

Step 3: Up
2 _ 3
4 1 6
5 7 8

Step 4: Left
_ 2 3
4 1 6
5 7 8

Step 5: Down
4 2 3
_ 1 6
5 7 8

Step 6: Down
4 2 3
5 1 6
_ 7 8

Step 7: Right
4 2 3
5 1 6
7 _ 8

Step 8: Right
4 2 3
5 1 6
7 8 _

Step 9: Up
4 2 3
5 1 _
7 8 6

Step 10: Up
4 2 _
5 1 3
7 8 6

Step 11: Left
4 _ 2
5 1 3
7 8 6

Step 12: Down
4 1 2
5 _ 3
7 8 6

Step 13: Left
4 1 2
_ 5 3
7 8 6

Step 14: Up
_ 1 2
4 5 3
7 8 6

Step 15: Right
1 _ 2
4 5 3
7 8 6

Step 16: Right
1 2 _
4 5 3
7 8 6

Step 17: Down
1 2 3
4 5 _
7 8 6

Step 18: Down
1 2 3
4 5 6
7 8 _

